<a href="https://colab.research.google.com/github/Elenadr/entrenamientoLLM_iot/blob/main/LLMFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalación de dependencias y conexión con Google Drive

In [1]:
import os
import re
import glob
import random
import pandas as pd
import numpy as np
from google.colab import drive
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader, Dataset
import torch
from sklearn.metrics import f1_score, precision_score, recall_score

import uuid

# Montar Google Drive
drive.mount('/content/drive')

# Instalar dependencias
!pip install transformers datasets torch pandas numpy

# Definir clases
sub_classes = ["Frecuencia Insegura", "Sin Cifrado", "Modulación Insegura", "Código Fijo", "Segura"]
nfc_classes = ["Claves por defecto", "Lectura completa", "Escritura posible", "UID clonable", "No detectada"]

# Definir claves por defecto
default_keys = {
    "FF FF FF FF FF FF",
    "A0 A1 A2 A3 A4 A5",
    "D3 F7 D3 F7 D3 F7",
    "00 00 00 00 00 00",
    "B0 B1 B2 B3 B4 B5",
    "4D 3A 99 C3 51 DD"
}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Etiquetado de señales SubGHz

In [2]:
def label_sub_file(content, debug=False):
    lines = content.split('\n')
    vulnerabilities = []
    frecuencia_hz = None
    preset = ""
    protocolo = ""
    key = ""
    has_raw_data = any("RAW_Data" in line for line in lines)

    for line in lines:
        if line.startswith("Frequency:"):
            try:
                frecuencia_hz = int(line.split(":")[1].strip())
            except ValueError:
                frecuencia_hz = None
        elif line.startswith("Preset:"):
            preset = line.split(":")[1].strip()
        elif line.startswith("Protocol:"):
            protocolo = line.split(":")[1].strip()
        elif line.startswith("Key:"):
            key = line.split(":", 1)[1].strip()

    if debug:
        print(f"Debug: Frequency={frecuencia_hz}, Preset={preset}, Protocol={protocolo}, Key={key}, Has RAW={has_raw_data}")

    secure_frequencies = [868350000, 915000000]
    secure_presets = ["FuriHalSubGhzPresetAES"]
    secure_protocols = ["AES-Encrypted", "KeeLoq", "KeeLoq-Secure"]
    prob_rolling_code = 0.0
    prob_codigo_fijo = 0.0

    # Frequency vulnerability
    if frecuencia_hz in [433920000, 315000000]:
        vulnerabilities.append("Frecuencia Insegura")
        prob_codigo_fijo += 0.3
        if debug:
            print("Debug: Added Frecuencia Insegura")

    # Protocol vulnerability
    insecure_protocols = ["OOK", "RAW"]
    if any(p.lower() in protocolo.lower() for p in insecure_protocols) and protocolo not in secure_protocols:
        vulnerabilities.append("Sin Cifrado")
        prob_codigo_fijo += 0.3
        if debug:
            print("Debug: Added Sin Cifrado")

    # Preset vulnerability
    insecure_presets = ["FuriHalSubGhzPresetOok650Async", "FuriHalSubGhzPresetOok270Async"]
    if preset in insecure_presets:
        vulnerabilities.append("Modulación Insegura")
        prob_codigo_fijo += 0.3
        if debug:
            print("Debug: Added Modulación Insegura")

    # Código Fijo and Rolling Code detection
    if has_raw_data:
        raw_lines = [line for line in lines if "RAW_Data" in line]
        raws = [re.findall(r"[-\d]+", line) for line in raw_lines]
        if len(raws) >= 2:
            repetition_count = sum(1 for i in range(len(raws)-1) if raws[i] == raws[i+1])
            if repetition_count >= len(raws) * 0.5:
                prob_codigo_fijo = max(prob_codigo_fijo, 1.0)
                if debug:
                    print("Debug: High repetition in RAW_Data, setting prob_codigo_fijo=1.0")
            elif len(raw_lines) > 10 and repetition_count == 0:
                prob_rolling_code = 0.7
                if debug:
                    print("Debug: Long non-repeating RAW_Data, setting prob_rolling_code=0.7")
    else:
        if protocolo in ["KeeLoq", "KeeLoq-Secure"]:
            prob_rolling_code = 0.9
            if debug:
                print("Debug: KeeLoq protocol, setting prob_rolling_code=0.9")
        else:
            key_parts = key.split()
            if len(key_parts) > 4 and len(set(key_parts)) < len(key_parts) / 2:
                prob_codigo_fijo = max(prob_codigo_fijo, 0.8)
                if debug:
                    print("Debug: Repetitive key pattern, setting prob_codigo_fijo=0.8")

    if prob_codigo_fijo >= 0.8:
        vulnerabilities.append("Código Fijo")
        if debug:
            print("Debug: Added Código Fijo")

    # Strict check for Segura
    if (not vulnerabilities and
        frecuencia_hz in secure_frequencies and
        preset in secure_presets and
        protocolo in secure_protocols and
        not has_raw_data):
        vulnerabilities = ["Segura"]
        if debug:
            print("Debug: No vulnerabilities and secure configuration, labeling as Segura")
    else:
        if prob_rolling_code >= 0.8 and debug:
            print("Debug: High prob_rolling_code but vulnerabilities present, keeping vulnerabilities")
        prob_rolling_code = min(prob_rolling_code, 0.2) if vulnerabilities else prob_rolling_code
        if debug:
            print("Debug: Vulnerabilities detected, adjusting prob_rolling_code")

    if debug:
        print(f"Debug: Final vulnerabilities={vulnerabilities}, Prob Rolling Code={prob_rolling_code}")

    return vulnerabilities, prob_rolling_code

## Etiquetado de señales NFC

In [3]:
def label_nfc_file(content, debug=False):
    lines = content.split('\n')
    vulnerable_uid = False
    bloques_leidos = 0
    claves_encontradas = set()
    tipo = None
    pages_read = 0
    pages_total = 0
    has_password = False

    for line in lines:
        if line.startswith("Device type:") or line.startswith("Type:"):
            tipo = line.strip().split(":", 1)[1].strip()
        elif re.search(r'Block \d+:', line) and "MIFARE Classic" in tipo:
            bloques_leidos += 1
            data = line.split(':', 1)[1].strip().upper()
            if any(key in data for key in default_keys):
                claves_encontradas.add(data)
        elif re.search(r'Key [AB]:', line) and "MIFARE Classic" in tipo:
            key = line.split(':', 1)[1].strip().upper()
            if any(key in dk for dk in default_keys):
                claves_encontradas.add(key)
        elif line.startswith("Pages total:"):
            pages_total = int(line.split(":")[1].strip())
        elif line.startswith("Pages read:"):
            pages_read = int(line.split(":")[1].strip())
        elif "Password:" in line or "PACK:" in line:
            has_password = True

    if debug:
        print(f"Debug: Type={tipo}, Bloques leídos={bloques_leidos}, Claves encontradas={claves_encontradas}, Pages read={pages_read}/{pages_total}, Has password={has_password}")

    vulnerabilidades = []
    if "MIFARE Classic" in tipo:
        vulnerable_uid = True  # Assume clonable unless locked
        if claves_encontradas:
            vulnerabilidades.append("Claves por defecto")
        if bloques_leidos >= 8:  # Umbral relajado para Lectura completa
            vulnerabilidades.append("Lectura completa")
        if claves_encontradas and bloques_leidos > 0:
            vulnerabilidades.append("Escritura posible")
    elif "NTAG" in tipo:
        if pages_read >= pages_total and pages_total > 0:
            vulnerabilidades.append("Lectura completa")
        if not has_password:
            vulnerabilidades.append("Escritura posible")
        vulnerable_uid = False
    else:
        vulnerable_uid = False

    if vulnerable_uid:
        vulnerabilidades.append("UID clonable")

    if not vulnerabilidades:
        vulnerabilidades = ["No detectada"]

    if debug:
        print(f"Debug: Vulnerabilidades={vulnerabilidades}")

    return vulnerabilidades

### Generación sintética de archivos NFC para el entrenamiento

In [4]:
def generate_synthetic_nfc_files(output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for f in glob.glob(os.path.join(output_folder, '*.nfc')):
        os.remove(f)

    for i in range(20):
        keys = random.choice(list(default_keys))
        content = f"""Filetype: Flipper NFC device
Version: 2
Device type: MIFARE Classic
UID: 04 {i:02X} {i+1:02X} 4{i}
Block 0: {keys}
Block 1: A0 A1 A2 A3 A4 A5
Key A: 00 00 00 00 00 00
Comment: Test {i}
"""
        with open(os.path.join(output_folder, f'synthetic_nfc_claves_{i}.nfc'), 'w') as f:
            f.write(content)

    for i in range(20):
        content = f"""Filetype: Flipper NFC device
Version: 2
Device type: MIFARE Classic
UID: 04 {i+20:02X} {i+21:02X} 5{i}
"""
        for j in range(16):
            content += f"Block {j}: {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X}\n"
        with open(os.path.join(output_folder, f'synthetic_nfc_lectura_{i}.nfc'), 'w') as f:
            f.write(content)

    for i in range(20):
        pages = random.choice([20, 45, 135])
        content = f"""Filetype: Flipper NFC device
Version: 2
Device type: NTAG215
UID: 04 {i+40:02X} {i+41:02X} C2 FC 67 80
Pages total: {pages}
Pages read: {pages}
"""
        with open(os.path.join(output_folder, f'synthetic_nfc_escritura_{i}.nfc'), 'w') as f:
            f.write(content)

    for i in range(20):
        content = f"""Filetype: Flipper NFC device
Version: 2
Device type: MIFARE Classic
UID: 04 {i+60:02X} {i+61:02X} 6{i}
Block 0: 12 34 56 78 90 AB
"""
        with open(os.path.join(output_folder, f'synthetic_nfc_uid_{i}.nfc'), 'w') as f:
            f.write(content)

    for i in range(20):
        device_type = random.choice(["MIFARE DESFire", "NTAG215"])
        if device_type == "MIFARE DESFire":
            content = f"""Filetype: Flipper NFC device
Version: 2
Device type: MIFARE DESFire
UID: 04 {i+80:02X} {i+81:02X} 7{i}
Protocol: AES-Encrypted
"""
        else:
            content = f"""Filetype: Flipper NFC device
Version: 2
Device type: NTAG215
UID: 04 {i+80:02X} {i+81:02X} 7{i}
Pages total: 45
Pages read: 45
Password: FF FF FF FF
PACK: 00 00
"""
        with open(os.path.join(output_folder, f'synthetic_nfc_secure_{i}.nfc'), 'w') as f:
            f.write(content)


### Generación sintética de archivos SubGHz para el entrenamiento

In [5]:
def generate_synthetic_sub_files(output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for f in glob.glob(os.path.join(output_folder, '*.sub')):
        os.remove(f)

    # Secure AES-Encrypted
    for i in range(15):
        content = f"""Filetype: Flipper SubGhz Key File
Version: 1
Frequency: 915000000
Preset: FuriHalSubGhzPresetAES
Protocol: AES-Encrypted
Bit: 128
Key: {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {i:02X}
Manufacture: Secure-Tech
"""
        with open(os.path.join(output_folder, f'synthetic_sub_aes_secure_{i}.sub'), 'w') as f:
            f.write(content)

    # Vulnerable fixed code with RAW data
    for i in range(30):
        content = f"""Filetype: Flipper SubGhz RAW File
Version: 1
Frequency: 315000000
Preset: FuriHalSubGhzPresetOok650Async
Protocol: RAW
RAW_Data: -39000 1000 -1000 3000 -1000 3000 -1000 3000 -1000 3000 -1000 3000 -1000 3000 -1000 3000
RAW_Data: -39000 1000 -1000 3000 -1000 3000 -1000 3000 -1000 3000 -1000 3000 -1000 3000 -1000 3000
"""
        with open(os.path.join(output_folder, f'synthetic_sub_raw_fixed_{i}.sub'), 'w') as f:
            f.write(content)

    # Vulnerable fixed code with key
    for i in range(30):
        content = f"""Filetype: Flipper SubGhz Key File
Version: 1
Frequency: 433920000
Preset: FuriHalSubGhzPresetOok650Async
Protocol: OOK
Bit: 64
Key: 12 34 56 78 12 34 56 78
Manufacture: Unknown
"""
        with open(os.path.join(output_folder, f'synthetic_sub_fixed_key_{i}.sub'), 'w') as f:
            f.write(content)

    # KeeLoq with vulnerabilities
    for i in range(30):
        content = f"""Filetype: Flipper SubGhz Key File
Version: 1
Frequency: 433920000
Preset: FuriHalSubGhzPresetOok270Async
Protocol: KeeLoq-Secure
Bit: 64
Key: {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {i:02X}
Manufacture: AdvancedSecurity
"""
        with open(os.path.join(output_folder, f'synthetic_sub_keeloq_vuln_{i}.sub'), 'w') as f:
            f.write(content)

    # KeeLoq with secure frequency but insecure modulation
    for i in range(20):
        content = f"""Filetype: Flipper SubGhz Key File
Version: 1
Frequency: 868350000
Preset: FuriHalSubGhzPresetOok270Async
Protocol: KeeLoq-Secure
Bit: 64
Key: {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {random.randint(0, 255):02X} {i:02X}
Manufacture: AdvancedSecurity
"""
        with open(os.path.join(output_folder, f'synthetic_sub_keeloq_modulacion_{i}.sub'), 'w') as f:
            f.write(content)




### Etiquetar y mover archivos nfc

In [ ]:
def label_and_move_nfc_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    labeled_files = []
    for archivo in os.listdir(input_folder):
        if archivo.endswith(".nfc"):
            input_path = os.path.join(input_folder, archivo)
            with open(input_path, 'r', encoding='utf-8') as f:
                content = f.read()
            vulnerabilities = label_nfc_file(content, debug=True)
            labeled_files.append((archivo, vulnerabilities))
            etiqueta = "# Vulnerabilidad: " + ", ".join(vulnerabilities)
            nuevo_contenido = etiqueta + "\n" + content
            output_path = os.path.join(output_folder, archivo)
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(nuevo_contenido)
            os.remove(input_path)
    return labeled_files

### Etiquetar y mover archivos .sub

In [ ]:

def label_and_move_sub_files(input_folder, output_folder, debug=False):
    os.makedirs(output_folder, exist_ok=True)
    labeled_files = []
    for archivo in os.listdir(input_folder):
        if archivo.endswith(".sub"):
            input_path = os.path.join(input_folder, archivo)
            with open(input_path, 'r', encoding='utf-8') as f:
                content = f.read()
            vulnerabilities, prob_rolling_code = label_sub_file(content, debug=debug)
            labeled_files.append((archivo, vulnerabilities, prob_rolling_code))
            etiqueta = "# Vulnerabilidad: " + ", ".join(vulnerabilities) + (f", Alta probabilidad de Código Variable (Rolling Code)" if prob_rolling_code >= 0.8 else "")
            nuevo_contenido = etiqueta + "\n" + content
            output_path = os.path.join(output_folder, archivo)
            output_path = get_unique_filename(output_path)
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(nuevo_contenido)
            os.remove(input_path)
    return labeled_files



### Downsample Modulación Insegura

In [ ]:

def downsample_modulacion_insegura(output_folder, labeled_files, max_files_to_remove=20):
    print("\nDownsampling Modulación Insegura y Sin Cifrado...")
    files_to_remove = []
    modulacion_files = [(f, v, p) for f, v, p in labeled_files if "Modulación Insegura" in v]
    sin_cifrado_files = [(f, v, p) for f, v, p in labeled_files if "Sin Cifrado" in v and "Código Fijo" not in v]

    files_to_remove.extend([f for f, _, _ in random.sample(modulacion_files, min(10, len(modulacion_files)))])
    files_to_remove.extend([f for f, _, _ in random.sample(sin_cifrado_files, min(10, len(sin_cifrado_files)))])

    for archivo in files_to_remove[:max_files_to_remove]:
        print(f"Eliminando: {os.path.basename(archivo)}")
        try:
            os.remove(archivo)
        except FileNotFoundError:
            print(f"Archivo {os.path.basename(archivo)} no encontrado, ya eliminado o movido")

    print(f"Eliminados {len(files_to_remove[:max_files_to_remove])} archivos")

### Carga y distribución de archivos

In [7]:
def load_files_for_distribution(folder, file_extension):
    data = []
    problematic_files = []

    files = glob.glob(os.path.join(folder, f'*{file_extension}'))
    if not files:
        print(f"Advertencia: No se encontraron archivos {file_extension} en {folder}")

    for file_path in files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                if not lines:
                    problematic_files.append((file_path, "Archivo vacio"))
                    continue

                content = ''.join(lines)
                if file_extension == '.sub':
                    vulnerabilities, prob_rolling_code = label_sub_file(content, debug=True)
                    data.append({
                        'filename': os.path.basename(file_path),
                        'vulnerabilities': ', '.join(vulnerabilities),
                        'prob_rolling_code': prob_rolling_code
                    })
                else:
                    vulnerabilities = label_nfc_file(content, debug=True)
                    data.append({
                        'filename': os.path.basename(file_path),
                        'vulnerabilities': ', '.join(vulnerabilities)
                    })
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='latin-1') as f:
                    content = ''.join(f.readlines())
                if file_extension == '.sub':
                    vulnerabilities, prob_rolling_code = label_sub_file(content, debug=True)
                    data.append({
                        'filename': os.path.basename(file_path),
                        'vulnerabilities': ', '.join(vulnerabilities),
                        'prob_rolling_code': prob_rolling_code
                    })
                else:
                    vulnerabilities = label_nfc_file(content, debug=True)
                    data.append({
                        'filename': os.path.basename(file_path),
                        'vulnerabilities': ', '.join(vulnerabilities)
                    })
            except Exception as e:
                problematic_files.append((file_path, f"Error al procesar con latin-1: {str(e)}"))
        except Exception as e:
            problematic_files.append((file_path, str(e)))

    if problematic_files:
        print("Archivos problematicos en distribucion:")
        for file_path, error in problematic_files:
            print(f"{file_path}: {error}")

    df = pd.DataFrame(data)
    if df.empty:
        print(f"Advertencia: No se generaron datos para {file_extension} en {folder}")
    else:
        print(f"Se procesaron {len(df)} archivos {file_extension} en {folder}")

    return df


## Soft labels para SUB

In [ ]:
def get_soft_labels(vulnerabilities, prob_rolling_code, classes):
    labels = np.zeros(len(classes))
    vuln_list = [v.strip() for v in vulnerabilities.split(',') if v.strip()]
    if vuln_list == ["Segura"]:
        labels[classes.index("Segura")] = 1.0
        return labels
    for vuln in vuln_list:
        if vuln in classes and vuln != "Segura":
            labels[classes.index(vuln)] = 1.0
    if any(v in vuln_list for v in ["Frecuencia Insegura", "Sin Cifrado", "Modulación Insegura", "Código Fijo"]):
        labels[classes.index("Segura")] = 0.0
    return labels

## Hard labels para NFC

In [8]:

def get_hard_labels(vulnerabilities, classes):
    labels = np.zeros(len(classes))
    vuln_list = [v.strip() for v in vulnerabilities.split(',') if v.strip()]
    for vuln in vuln_list:
        if vuln in classes:
            labels[classes.index(vuln)] = 1.0
    return labels



### Comprobación de etiquetas

In [ ]:
def check_label_distribution(df, classes, file_type):
    if df.empty:
        print(f"Error: DataFrame vacio para {file_type}. No se puede calcular la distribucion de clases.")
        return

    if file_type == '.sub' and 'prob_rolling_code' not in df.columns:
        print("Error: Columna 'prob_rolling_code' no encontrada en DataFrame para .sub")
        return

    if 'vulnerabilities' not in df.columns:
        print("Error: Columna 'vulnerabilities' no encontrada en DataFrame")
        return

    if file_type == '.sub':
        df['labels'] = df.apply(lambda x: get_soft_labels(x['vulnerabilities'], x['prob_rolling_code'], classes), axis=1)
    else:
        df['labels'] = df['vulnerabilities'].apply(lambda x: get_hard_labels(x, classes))

    label_counts = np.zeros(len(classes))
    for labels in df['labels']:
        label_counts += np.array(labels)
    print(f"\nDistribución de clases {file_type}:")
    for i, cls in enumerate(classes):
        print(f"{cls}: {label_counts[i]:.2f} instancias")

### Obtener nombre de archivo único

In [ ]:
def get_unique_filename(output_path):
    base, ext = os.path.splitext(output_path)
    counter = 1
    new_path = output_path
    while os.path.exists(new_path):
        new_path = f"{base}_{counter}{ext}"
        counter += 1
    return new_path

### Cargar archivos para entrenamiento

In [9]:
def load_files(folder, file_extension):
    data = []
    problematic_files = []

    for file_path in glob.glob(os.path.join(folder, f'*{file_extension}')):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                if not lines:
                    problematic_files.append((file_path, "Archivo vacío"))
                    continue

                content = ''.join(lines)
                if file_extension == '.sub':
                    vulnerabilities, prob_rolling_code = label_sub_file(content, debug=True)
                    data.append({
                        'filename': os.path.basename(file_path),
                        'content': content,
                        'vulnerabilities': ', '.join(vulnerabilities),
                        'prob_rolling_code': prob_rolling_code
                    })
                else:
                    vulnerabilities = label_nfc_file(content, debug=True)
                    data.append({
                        'filename': os.path.basename(file_path),
                        'content': content,
                        'vulnerabilities': ', '.join(vulnerabilities)
                    })
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='latin-1') as f:
                    lines = f.readlines()
                    content = ''.join(lines)
                    if file_extension == '.sub':
                        vulnerabilities, prob_rolling_code = label_sub_file(content, debug=True)
                        data.append({
                            'filename': os.path.basename(file_path),
                            'content': content,
                            'vulnerabilities': ', '.join(vulnerabilities),
                            'prob_rolling_code': prob_rolling_code
                        })
                    else:
                        vulnerabilities = label_nfc_file(content, debug=True)
                        data.append({
                            'filename': os.path.basename(file_path),
                            'content': content,
                            'vulnerabilities': ', '.join(vulnerabilities)
                        })
            except Exception as e:
                problematic_files.append((file_path, f"Error al procesar con latin-1: {str(e)}"))
        except Exception as e:
            problematic_files.append((file_path, str(e)))

    if problematic_files:
        print("Archivos problemáticos:")
        for file_path, error in problematic_files:
            print(f"{file_path}: {error}")

    return pd.DataFrame(data)


### Preparar datos para entrenamiento

In [10]:

def prepare_data(df, classes, file_type, tokenizer):
    texts = df['content'].tolist()
    encodings = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

    if file_type == '.sub':
        labels = [get_soft_labels(row['vulnerabilities'], row['prob_rolling_code'], classes) for _, row in df.iterrows()]
    else:
        labels = [get_hard_labels(row['vulnerabilities'], classes) for _, row in df.iterrows()]

    data_dict = {
        'input_ids': encodings['input_ids'].tolist(),
        'attention_mask': encodings['attention_mask'].tolist(),
        'labels': labels
    }
    if 'token_type_ids' in encodings:
        data_dict['token_type_ids'] = encodings['token_type_ids'].tolist()

    return data_dict





### Evaluar modelo

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.3).int().numpy()
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    return {
        'f1_macro': f1,
        'precision_macro': precision,
        'recall_macro': recall
    }

In [ ]:
def evaluate_model(trainer, dataset, classes):
    print("\nEvaluación del modelo NFC:")
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.3).int().numpy()
    true_labels = np.array([item['labels'] for item in dataset])

    for i, cls in enumerate(classes):
        f1 = f1_score(true_labels[:, i], preds[:, i], zero_division=0)
        print(f"F1-score para {cls}: {f1:.4f}")

    print(f"F1-score macro: {f1_score(true_labels, preds, average='macro', zero_division=0):.4f}")
    print(f"Precisión macro: {precision_score(true_labels, preds, average='macro', zero_division=0):.4f}")
    print(f"Recall macro: {recall_score(true_labels, preds, average='macro', zero_division=0):.4f}")

# Función para predecir vulnerabilidades con explotación

In [11]:

def predict_vulnerabilities(file_path, threshold=0.3):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
    except UnicodeDecodeError:
        with open(file_path, 'r', encoding='latin-1') as f:
            content = f.read().strip()
    except Exception as e:
        print(f"Error al leer {file_path}: {str(e)}")
        return ["Error al procesar el archivo"]
    model = sub_model if file_path.endswith('.sub') else nfc_model
    classes = sub_classes if file_path.endswith('.sub') else nfc_classes

    # Extraer atributos
    frecuencia_hz = None
    preset = ""
    protocolo = ""
    tipo = ""
    has_raw_data = "RAW_Data" in content
    has_password = "Password:" in content or "PACK:" in content
    bloques_leidos = len([line for line in content.split('\n') if re.search(r'Block \d+:', line)])
    claves_encontradas = any(key in content.upper() for key in default_keys)

    for line in content.split('\n'):
        if line.startswith("Frequency:"):
            try:
                frecuencia_hz = int(line.split(":")[1].strip())
            except ValueError:
                frecuencia_hz = None
        elif line.startswith("Preset:"):
            preset = line.split(":")[1].strip()
        elif line.startswith("Protocol:"):
            protocolo = line.split(":")[1].strip()
        elif line.startswith("Device type:") or line.startswith("Type:"):
            tipo = line.strip().split(":", 1)[1].strip()

    # Añadir características extraídas
    extra_features = f"Bloques leidos: {bloques_leidos} Claves encontradas: {claves_encontradas} Tipo: {tipo}"
    content_with_features = content + "\n" + extra_features

    encodings = tokenizer(content_with_features, padding=True, truncation=True, max_length=512, return_tensors='pt')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    encodings = {key: val.to(device) for key, val in encodings.items()}
    model.to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # Sigmoid para multi-label

    print(f"\nProbabilidades para {file_path}:")
    for cls, prob in zip(classes, probs):
        print(f"{cls}: {prob:.4f}")

    predicted_vulns = [classes[i] for i, prob in enumerate(probs) if prob > threshold]

    # Reglas basadas en características extraídas para NFC
    if file_path.endswith('.nfc'):
        if claves_encontradas and "Claves por defecto" not in predicted_vulns:
            predicted_vulns.append("Claves por defecto")
        if bloques_leidos >= 8 and "Lectura completa" not in predicted_vulns:
            predicted_vulns.append("Lectura completa")
        if "MIFARE Classic" in tipo and "UID clonable" not in predicted_vulns:
            predicted_vulns.append("UID clonable")

    final_vulns = predicted_vulns.copy()
    if file_path.endswith('.sub'):
        if any(v in final_vulns for v in ["Frecuencia Insegura", "Sin Cifrado", "Modulación Insegura", "Código Fijo"]):
            if "Segura" in final_vulns:
                final_vulns.remove("Segura")
    else:
        if any(v in final_vulns for v in ["Claves por defecto", "Lectura completa", "Escritura posible", "UID clonable"]):
            if "No detectada" in final_vulns:
                final_vulns.remove("No detectada")

    if not final_vulns:
        final_vulns = ["Ninguna vulnerabilidad detectada"]

    # Formatear salida
    print("\n" + "="*50)
    print(f"📄 Análisis de: {file_path}")
    print("="*50)
    print("\n🚨 Vulnerabilidades detectadas:")
    print("-"*40)
    for vuln in final_vulns:
        print(f"• {vuln}")
    if file_path.endswith('.sub'):
        _, prob_rolling_code = label_sub_file(content, debug=True)
        print(f"• Probabilidad de Código Variable: {prob_rolling_code:.2f}")
    print("-"*40)

    # Sugerencias de explotación
    if final_vulns != ["Ninguna vulnerabilidad detectada"]:
        print("\n💥 Métodos de explotación sugeridos:")
        print("-"*40)
        if file_path.endswith('.sub'):
            for vuln in final_vulns:
                if vuln == "Frecuencia Insegura":
                    print(f"🔧 {vuln}:")
                    print(f"   Frecuencia: {frecuencia_hz} Hz")
                    print("   Explotación: Capturar y retransmitir la señal para un ataque de repetición.")
                    print("   Herramientas: Flipper Zero, HackRF One, Yard Stick One")
                elif vuln == "Sin Cifrado":
                    print(f"🔧 {vuln}:")
                    print(f"   Protocolo: {protocolo}")
                    print("   Explotación: Interceptar y decodificar datos en texto plano.")
                    print("   Herramientas: URH, Flipper Zero, RTL-SDR")
                elif vuln == "Modulación Insegura":
                    print(f"🔧 {vuln}:")
                    print(f"   Preset: {preset}")
                    print("   Explotación: Jamming o suplantación de señal debido a modulación predecible.")
                    print("   Herramientas: Flipper Zero, HackRF One")
                elif vuln == "Código Fijo":
                    print(f"🔧 {vuln}:")
                    print("   Explotación: Clonar código fijo capturando y retransmitiéndolo.")
                    print("   Herramientas: Flipper Zero, Proxmark3, SDR")
                elif vuln == "Segura":
                    print(f"🔧 {vuln}:")
                    print("   Nota: Protocolo seguro detectado. Verificar implementación del receptor.")
        else:
            for vuln in final_vulns:
                if vuln == "Claves por defecto":
                    print(f"🔧 {vuln}:")
                    print(f"   Tipo: {tipo}")
                    print("   Explotación: Usar claves por defecto para acceder a datos o modificar tarjeta.")
                    print("   Herramientas: Proxmark3, NFC Tools")
                elif vuln == "Lectura completa":
                    print(f"🔧 {vuln}:")
                    print(f"   Bloques leídos: {bloques_leidos}")
                    print("   Explotación: Extraer todos los datos de la tarjeta para análisis o clonación.")
                    print("   Herramientas: Proxmark3, NFC Tools")
                elif vuln == "Escritura posible":
                    print(f"🔧 {vuln}:")
                    print("   Explotación: Modificar datos de la tarjeta para alterar su comportamiento.")
                    print("   Herramientas: Proxmark3, NFC Tools")
                elif vuln == "UID clonable":
                    print(f"🔧 {vuln}:")
                    print(f"   Tipo: {tipo}")
                    print("   Explotación: Clonar UID para replicar tarjeta.")
                    print("   Herramientas: Proxmark3, Flipper Zero")
                elif vuln == "No detectada":
                    print(f"🔧 {vuln}:")
                    print("   Nota: No se detectaron vulnerabilidades. Verificar configuración física.")
        print("-"*40)

    print("\n" + "="*50 + "\n")
    return final_vulns


### Ejecutar generación, etiquetado y entrenamiento

In [ ]:
# Definir rutas de carpetas
synthetic_nfc_folder = '/content/drive/MyDrive/ParaEtiquetarNFC'
synthetic_sub_folder = '/content/drive/MyDrive/Dataset/sub_files_synthetic'
sub_input_folder = '/content/drive/MyDrive/ParaEtiquetar'
sub_output_folder = '/content/drive/MyDrive/EtiquetadasSub'
nfc_output_folder = '/content/drive/MyDrive/EtiquetadasNFC'

# Generar archivos sintéticos
generate_synthetic_nfc_files(synthetic_nfc_folder)
generate_synthetic_sub_files(synthetic_sub_folder)

# Etiquetar y mover archivos
sub_labeled_synthetic = label_and_move_sub_files(synthetic_sub_folder, sub_output_folder, debug=True)
sub_labeled_input = label_and_move_sub_files(sub_input_folder, sub_output_folder, debug=True)
nfc_labeled = label_and_move_nfc_files(synthetic_nfc_folder, nfc_output_folder)

# Downsample Modulación Insegura
downsample_modulacion_insegura(sub_output_folder, sub_labeled_synthetic + sub_labeled_input)

# Verificar distribución de clases
sub_df = load_files_for_distribution(sub_output_folder, '.sub')
nfc_df = load_files_for_distribution(nfc_output_folder, '.nfc')
check_label_distribution(sub_df, sub_classes, '.sub')
check_label_distribution(nfc_df, nfc_classes, '.nfc')

# Cargar datos para entrenamientoif labels.shape[1]
sub_df_train = load_files(sub_output_folder, '.sub')
nfc_df_train = load_files(nfc_output_folder, '.nfc')

# Inicializar tokenizer y modelos
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
sub_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(sub_classes))
nfc_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(nfc_classes), problem_type="multi_label_classification")
# Preparar datasets
sub_data_dict = prepare_data(sub_df_train, sub_classes, '.sub', tokenizer)
nfc_data_dict = prepare_data(nfc_df_train, nfc_classes, '.nfc', tokenizer)
# Convertir a Hugging Face Dataset y dividir
sub_dataset = Dataset.from_dict(sub_data_dict).train_test_split(test_size=0.2)
nfc_dataset = Dataset.from_dict(nfc_data_dict).train_test_split(test_size=0.2)


Se han truncado las últimas 5000 líneas del flujo de salida.
Debug: Final vulnerabilities=['Frecuencia Insegura', 'Sin Cifrado'], Prob Rolling Code=0.0
Debug: Frequency=868350000, Preset=FuriHalSubGhzPresetAES, Protocol=AES-Encrypted, Key=1A 2B 3C 4D 5E 6F 70 00, Has RAW=False
Debug: No vulnerabilities and secure configuration, labeling as Segura
Debug: Final vulnerabilities=['Segura'], Prob Rolling Code=0.0
Debug: Frequency=868350000, Preset=FuriHalSubGhzPresetAES, Protocol=AES-Encrypted, Key=1A 2B 3C 4D 5E 6F 70 01, Has RAW=False
Debug: No vulnerabilities and secure configuration, labeling as Segura
Debug: Final vulnerabilities=['Segura'], Prob Rolling Code=0.0
Debug: Frequency=868350000, Preset=FuriHalSubGhzPresetAES, Protocol=AES-Encrypted, Key=1A 2B 3C 4D 5E 6F 70 02, Has RAW=False
Debug: No vulnerabilities and secure configuration, labeling as Segura
Debug: Final vulnerabilities=['Segura'], Prob Rolling Code=0.0
Debug: Frequency=868350000, Preset=FuriHalSubGhzPresetAES, Protocol=

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Entrenamiento del modelo

In [13]:
from datasets import Dataset
# Convertir a Hugging Face Dataset y dividir
sub_dataset = Dataset.from_dict(sub_data_dict).train_test_split(test_size=0.2)
nfc_dataset = Dataset.from_dict(nfc_data_dict).train_test_split(test_size=0.2)

# Configurar argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=5e-5,
)

# Entrenar modelo para .sub
sub_trainer = Trainer(
    model=sub_model,
    args=training_args,
    train_dataset=sub_dataset['train'],
    eval_dataset=sub_dataset['test'],
    compute_metrics=compute_metrics,
)
sub_trainer.train()

# Entrenar modelo para .nfc
nfc_trainer = Trainer(
    model=nfc_model,
    args=training_args,
    train_dataset=nfc_dataset['train'],
    eval_dataset=nfc_dataset['test'],
    compute_metrics=compute_metrics,
)
nfc_trainer.train()

# Evaluar modelos
evaluate_model(sub_trainer, sub_dataset['test'], sub_classes)
evaluate_model(nfc_trainer, nfc_dataset['test'], nfc_classes)

# Guardar modelos
sub_model.save_pretrained('/content/drive/MyDrive/models/sub_model')
nfc_model.save_pretrained('/content/drive/MyDrive/models/nfc_model')
tokenizer.save_pretrained('/content/drive/MyDrive/models/tokenizer')



wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: detodoapuntes0 (detodoapuntes0-viu-universidad-internacional-de-valencia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Epoch,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro
1,0.698400,0.673622,0.296970,0.259368,0.347368
2,0.631000,0.547818,0.269565,0.211329,0.400000
3,0.533000,0.489351,0.274255,0.229870,0.400000
4,0.486400,0.415966,0.437674,0.432727,0.533333
5,0.372200,0.325072,0.692308,0.660000,0.800000



Evaluación del modelo:


F1-score para Frecuencia Insegura: 0.8918
F1-score para Sin Cifrado: 0.0000
F1-score para Modulación Insegura: 0.8889
F1-score para Código Fijo: 0.0000
F1-score para Segura: 0.9633
F1-score macro: 0.6860
Precisión macro: 0.6823
Recall macro: 0.7012

Evaluación del modelo:


F1-score para Claves por defecto: 0.0000
F1-score para Lectura completa: 1.0000
F1-score para Escritura posible: 1.0000
F1-score para UID clonable: 0.4615
F1-score para No detectada: 1.0000
F1-score macro: 0.6923
Precisión macro: 0.6600
Recall macro: 0.8000


('/content/drive/MyDrive/models/tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/models/tokenizer/special_tokens_map.json',
 '/content/drive/MyDrive/models/tokenizer/vocab.txt',
 '/content/drive/MyDrive/models/tokenizer/added_tokens.json')

In [ ]:
# Testear archivos problemáticos
test_files = [
    "/content/drive/MyDrive/ParaEtiquetar/signal_15.sub",
    "/content/drive/MyDrive/ParaEtiquetar/Chamberlain-9bit-315-21.sub",
    "/content/drive/MyDrive/random/fixed_code_vulnerable.sub",
    "/content/drive/MyDrive/NFC/vuln_tag_10 (1).nfc"
]
for file in test_files:
    predict_vulnerabilities(file)


## Guardar modelos

In [ ]:
sub_model.save_pretrained('/content/drive/MyDrive/models/sub_model')
nfc_model.save_pretrained('/content/drive/MyDrive/models/nfc_model')
tokenizer.save_pretrained('/content/drive/MyDrive/models/tokenizer')